# poke-env経験者向け: poke-env互換プロパティ・APIを使った choose_command() の実装方法を示す

01_getting_started/03_custom_player.ipynb の CustomePlayer は available_commands() +
command_to_move() の組み合わせで実装していたが、poke-env互換プロパティ・
create_order()とpoke-envのBattleOrderとの違いは docs/api/README.md の
Battle「poke-env互換プロパティ」節・create_order()の項を参照。

Google Colabで開いた場合は、まず次のセルで `jpoke` をインストールする
（ローカルで `pip install jpoke` 済みなら不要）。

In [ ]:
%pip install -q jpoke

In [ ]:
from jpoke import Battle, Player
from jpoke.enums import Command

In [ ]:
class PokeEnvStylePlayer(Player):
    """poke-env互換プロパティを使い、最も威力の高い技を選ぶプレイヤー。"""

    def choose_command(self, battle: Battle) -> Command:
        moves = battle.available_moves  # available_commands()を技だけ取り出しMoveに変換したもの
        if not moves:
            # わるあがきのみの場合の available_moves / is_struggle_only() の挙動は
            # docs/api/README.md の該当節を参照
            assert battle.is_struggle_only(self)
            return Command.STRUGGLE
        best_move = max(moves, key=lambda m: m.base_power if m.is_attack else 0)
        return battle.create_order(self, best_move)

In [ ]:
player1 = PokeEnvStylePlayer("PokeEnvStylePlayer")
player1.add_pokemon("ピカチュウ", moves=["でんこうせっか", "かみなり", "なきごえ"])

player2 = Player("Player 2")
player2.add_pokemon("フシギダネ", moves=["たいあたり"])

In [ ]:
battle = Battle(player1, player2, seed=1)
battle.play_out(max_turns=100)
winner = battle.winner
print(f"勝者: {winner.username if winner else '引き分け（ターン上限）'}")

In [ ]:
active = battle.get_active(player1)
print(f"最終ターンの技: {active.last_move.name if active.last_move else None}")